# README-Compatible Adapter Inference Demo

このNotebookは README.md に記載された評価条件に合わせて、Nemotron-3-Nano-30B base model + 外部LoRA adapter を vLLM で読み込み、family ごとに数サンプルだけ推論して出力内容を確認するためのものです。

- 推論エンジン: vLLM
- LoRA 読み込み: LoRARequest
- 推論パラメータ: README.md 記載値に固定
- 目的: 精度計測ではなく raw output と boxed answer の確認

In [ ]:
import os
import re
from pathlib import Path

import kagglehub
import pandas as pd
import polars as pl

MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
ADAPTER_PATH = "/kaggle/input/your-trained-rule-based-adapter"
DATASET_CSV = "/kaggle/input/datasets/renta0426/rule-based-training-data/rule_based_verified_600_training_data.csv"
SAMPLES_PER_FAMILY = 10
OTHER_SAMPLES_PER_FAMILY = 2

README_MAX_LORA_RANK = 32
README_MAX_TOKENS = 7680
README_TOP_P = 1.0
README_TEMPERATURE = 0.0
README_MAX_NUM_SEQS = 64
README_GPU_MEMORY_UTILIZATION = 0.85
README_MAX_MODEL_LEN = 8192

adapter_path = Path(ADAPTER_PATH)
required_files = [adapter_path / "adapter_config.json", adapter_path / "adapter_model.safetensors"]
missing = [str(path) for path in required_files if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Adapter directory must contain adapter_config.json and adapter_model.safetensors. Missing: "
        + ", ".join(missing)
    )

print(f"MODEL_PATH={MODEL_PATH}")
print(f"ADAPTER_PATH={adapter_path}")
print(f"DATASET_CSV={DATASET_CSV}")

In [ ]:
CSV_SCHEMA = {
    "id": pl.Utf8,
    "prompt": pl.Utf8,
    "answer": pl.Utf8,
    "generated_cot": pl.Utf8,
    "label": pl.Utf8,
}

frame = pl.read_csv(DATASET_CSV, schema_overrides=CSV_SCHEMA)
family_counts = frame.group_by("label").len().sort("label")
display(family_counts)

binary_frame = (
    frame
    .filter(pl.col("label") == "binary")
    .sort("id")
    .head(SAMPLES_PER_FAMILY)
    .select(["label", "id", "prompt", "answer"])
)

other_frame = (
    frame
    .filter(pl.col("label") != "binary")
    .sort(["label", "id"])
)
other_frame = (
    other_frame
    .group_by("label", maintain_order=True)
    .head(OTHER_SAMPLES_PER_FAMILY)
    .select(["label", "id", "prompt", "answer"])
)

sample_frame = (
    pl.concat([binary_frame, other_frame])
    .sort(["label", "id"])
)

print(
    f"Selected {binary_frame.height} binary prompts and "
    f"{other_frame.height} prompts from other families"
 )
display(sample_frame)

In [ ]:
def extract_final_answer(text: str | None) -> str:
    if text is None:
        return "NOT_FOUND"

    matches = re.findall(r'\\boxed\{([^}]*)(?:\}|$)', text)
    if matches:
        non_empty = [item.strip() for item in matches if item.strip()]
        if non_empty:
            return non_empty[-1]
        return matches[-1].strip()

    patterns = [
        r'The final answer is:\s*([^\n]+)',
        r'Final answer is:\s*([^\n]+)',
        r'Final answer\s*[:：]\s*([^\n]+)',
        r'final answer\s*[:：]\s*([^\n]+)',
    ]
    for pattern in patterns:
        found = re.findall(pattern, text, re.IGNORECASE)
        if found:
            return found[-1].strip()

    numeric = re.findall(r'-?\d+(?:\.\d+)?', text)
    if numeric:
        return numeric[-1]

    lines = [line.strip() for line in text.splitlines() if line.strip()]
    return lines[-1] if lines else "NOT_FOUND"


def build_prompts(tokenizer, prompt_series: list[str]) -> list[str]:
    prompts = []
    for prompt_text in prompt_series:
        user_content = (
            prompt_text
            + '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'
        )
        try:
            rendered = tokenizer.apply_chat_template(
                [{"role": "user", "content": user_content}],
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=True,
            )
        except Exception:
            rendered = user_content
        prompts.append(rendered)
    return prompts

In [ ]:
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest

llm = LLM(
    model=str(MODEL_PATH),
    tensor_parallel_size=1,
    max_num_seqs=README_MAX_NUM_SEQS,
    gpu_memory_utilization=README_GPU_MEMORY_UTILIZATION,
    dtype='auto',
    max_model_len=README_MAX_MODEL_LEN,
    trust_remote_code=True,
    enable_lora=True,
    max_lora_rank=README_MAX_LORA_RANK,
    enable_prefix_caching=True,
    enable_chunked_prefill=True,
)

sampling_params = SamplingParams(
    temperature=README_TEMPERATURE,
    top_p=README_TOP_P,
    max_tokens=README_MAX_TOKENS,
)

tokenizer = llm.get_tokenizer()
prompts = build_prompts(tokenizer, sample_frame['prompt'].to_list())
outputs = llm.generate(
    prompts,
    sampling_params=sampling_params,
    lora_request=LoRARequest('adapter', 1, str(adapter_path)),
)

records = []
for row, output, rendered_prompt in zip(sample_frame.iter_rows(named=True), outputs, prompts):
    raw_output = output.outputs[0].text
    records.append({
        'label': row['label'],
        'id': row['id'],
        'expected_answer': row['answer'],
        'extracted_answer': extract_final_answer(raw_output),
        'rendered_prompt': rendered_prompt,
        'raw_output': raw_output,
    })

results_df = pd.DataFrame(records)
print(f"Generated {len(results_df)} samples with README-compatible vLLM settings")

In [ ]:
display(results_df[['label', 'id', 'expected_answer', 'extracted_answer']])

preview_df = results_df.copy()
preview_df['prompt_preview'] = preview_df['rendered_prompt'].str.slice(0, 220)
preview_df['raw_output_preview'] = preview_df['raw_output'].str.slice(0, 600)
display(preview_df[['label', 'id', 'prompt_preview', 'raw_output_preview']])

results_path = '/kaggle/working/rule_based_adapter_readme_inference_samples.csv'
results_df.to_csv(results_path, index=False)
print(f"Saved detailed outputs to {results_path}")

## Notes

- 精度は計算していません。目的は family 横断で出力品質を目視確認することです。
- 期待解は参考表示のみで、評価ロジックには使っていません。
- さらに見る件数を増やす場合は 3番目のセルの SAMPLES_PER_FAMILY を調整してください。
- adapter は外部参照前提なので、2番目のセルの ADAPTER_PATH を実配置に合わせて更新してください。